In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.conv(x)

class SimpleLLIE(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = self.conv_block(3, 32)
        self.enc2 = self.conv_block(32, 64)
        self.enc3 = self.conv_block(64, 128)

        # Decoder
        self.attn = AttentionBlock(128)
        self.dec2 = self.conv_block(128 + 64, 64)
        self.dec1 = self.conv_block(64 + 32, 32)

        self.final = nn.Conv2d(32, 3, 1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # UNet skip connections
        s1 = self.enc1(x)
        s2 = self.enc2(F.max_pool2d(s1, 2))
        s3 = self.enc3(F.max_pool2d(s2, 2))

        # Middle
        m = self.attn(s3)

        # Upsample & Concat
        u2 = F.interpolate(m, scale_factor=2, mode='bilinear', align_corners=True)
        u2 = self.dec2(torch.cat([u2, s2], dim=1))

        u1 = F.interpolate(u2, scale_factor=2, mode='bilinear', align_corners=True)
        u1 = self.dec1(torch.cat([u1, s1], dim=1))

        return torch.sigmoid(self.final(u1))

In [ ]:
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

class LLIE_Dataset(Dataset):
    def __init__(self, low_dir, high_dir, patch_size=256):
        self.files = sorted([f for f in os.listdir(low_dir) if f.endswith(('.png', '.jpg'))])
        self.low_dir = low_dir
        self.high_dir = high_dir
        self.patch_size = patch_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Load images
        low = cv2.imread(os.path.join(self.low_dir, self.files[idx]))
        high = cv2.imread(os.path.join(self.high_dir, self.files[idx]))

        # Get dimensions
        h, w, _ = low.shape
        ps = self.patch_size

        # 1. Select a random top-left corner for the crop
        # This ensures we get the same crop for both low and high images
        y = np.random.randint(0, h - ps)
        x = np.random.randint(0, w - ps)

        # 2. Crop both images at the same location
        low_patch = low[y:y+ps, x:x+ps, :]
        high_patch = high[y:y+ps, x:x+ps, :]

        # 3. Normalize and convert to tensors
        low_tensor = torch.from_numpy(low_patch / 255.0).permute(2,0,1).float()
        high_tensor = torch.from_numpy(high_patch / 255.0).permute(2,0,1).float()

        return low_tensor, high_tensor

# Initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleLLIE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.L1Loss() # Or Charbonnier Loss for better PSNR

# Training Loop (Example)
def train(epochs, dataloader):
    model.train()
    for epoch in range(epochs):
        for low, high in dataloader:
            low, high = low.to(device), high.to(device)
            output = model(low)
            loss = criterion(output, high)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1} Complete. Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), 'best_model.pth')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Define paths to your downloaded data
train_low_path = r'/content/drive/MyDrive/val_input'
train_high_path = r'/content/drive/MyDrive/val_reference'
# 2. Create Dataset and DataLoader
train_ds = LLIE_Dataset(train_low_path, train_high_path)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

# 3. Start Training (Aim for at least 50-100 epochs for initial results)
train(epochs=50, dataloader=train_loader)

Epoch 1 Complete. Loss: 0.2021
Epoch 2 Complete. Loss: 0.2983
Epoch 3 Complete. Loss: 0.1958
Epoch 4 Complete. Loss: 0.2163
Epoch 5 Complete. Loss: 0.3132
Epoch 6 Complete. Loss: 0.2476
Epoch 7 Complete. Loss: 0.2170
Epoch 8 Complete. Loss: 0.1763
Epoch 9 Complete. Loss: 0.2542
Epoch 10 Complete. Loss: 0.1632
Epoch 11 Complete. Loss: 0.2323
Epoch 12 Complete. Loss: 0.1711
Epoch 13 Complete. Loss: 0.1552
Epoch 14 Complete. Loss: 0.2184
Epoch 15 Complete. Loss: 0.1813
Epoch 16 Complete. Loss: 0.1919
Epoch 17 Complete. Loss: 0.1210
Epoch 18 Complete. Loss: 0.1421
Epoch 19 Complete. Loss: 0.1656
Epoch 20 Complete. Loss: 0.1604
Epoch 21 Complete. Loss: 0.1413
Epoch 22 Complete. Loss: 0.0791
Epoch 23 Complete. Loss: 0.2298
Epoch 24 Complete. Loss: 0.1437
Epoch 25 Complete. Loss: 0.1413
Epoch 26 Complete. Loss: 0.1381
Epoch 27 Complete. Loss: 0.1015
Epoch 28 Complete. Loss: 0.1401
Epoch 29 Complete. Loss: 0.1535
Epoch 30 Complete. Loss: 0.1437
Epoch 31 Complete. Loss: 0.1107
Epoch 32 Complete

In [ ]:
import torch
import gc

# Force release memory
torch.cuda.empty_cache()
gc.collect()

62

In [ ]:
import zipfile
from tqdm.notebook import tqdm

def generate_submission_zip(model_path, val_input_dir, zip_name="submission.zip", tile_size=256):
    model = SimpleLLIE().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    if not os.path.exists('temp_results'): os.makedirs('temp_results')
    files = sorted([f for f in os.listdir(val_input_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    with torch.no_grad():
        for f in tqdm(files, desc="Processing"):
            img_path = os.path.join(val_input_dir, f)
            img = cv2.imread(img_path)
            if img is None: continue

            img = img / 255.0
            h, w, c = img.shape
            # Create the result canvas on CPU (saves GPU VRAM)
            enhanced_img = np.zeros((h, w, c), dtype=np.float32)

            for y in range(0, h, tile_size):
                for x in range(0, w, tile_size):
                    y_end = min(y + tile_size, h)
                    x_end = min(x + tile_size, w)

                    tile = img[y:y_end, x:x_end, :]
                    t = torch.from_numpy(tile).permute(2, 0, 1).float().unsqueeze(0).to(device)

                    out = model(t)

                    # Move result back to CPU immediately
                    res_tile = out.squeeze().permute(1, 2, 0).cpu().numpy()
                    enhanced_img[y:y_end, x:x_end, :] = res_tile

                    # Clean up GPU
                    del t, out, res_tile

            final_res = (enhanced_img * 255.0).clip(0, 255).astype('uint8')
            cv2.imwrite(f'temp_results/{f}', final_res)

            # Clear cache after every full image
            torch.cuda.empty_cache()

    # Zip it up...

In [ ]:
import os
import cv2
import torch
import shutil
from tqdm.notebook import tqdm

In [ ]:
PAIR_INPUT_DIR = '/content/drive/MyDrive/T1_Pair_input'
UNPAIR_INPUT_DIR = '/content/drive/MyDrive/Unpair'
MODEL_PATH = 'best_model.pth'

In [ ]:
def run_ntire_inference(model_path, input_dir, output_dir, tile_size=256):
    """Runs inference and saves images with exact original names/resolutions."""
    model = SimpleLLIE().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    if not os.path.exists(output_dir): os.makedirs(output_dir)
    files = sorted([f for f in os.listdir(input_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    with torch.no_grad():
        for f in tqdm(files, desc=f"Processing {os.path.basename(input_dir)}"):
            img = cv2.imread(os.path.join(input_dir, f))
            if img is None: continue

            # Normalization and Tiled Inference
            img_norm = img / 255.0
            h, w, c = img_norm.shape
            enhanced_img = np.zeros((h, w, c), dtype=np.float32)

            for y in range(0, h, tile_size):
                for x in range(0, w, tile_size):
                    y_end = min(y + tile_size, h)
                    x_end = min(x + tile_size, w)

                    tile = img_norm[y:y_end, x:x_end, :]
                    t = torch.from_numpy(tile).permute(2, 0, 1).float().unsqueeze(0).to(device)

                    out = model(t)

                    res_tile = out.squeeze().permute(1, 2, 0).cpu().numpy()
                    enhanced_img[y:y_end, x:x_end, :] = res_tile
                    del t, out, res_tile

            # Save with original dimensions using LANCZOS equivalent (default imwrite)
            final_res = (enhanced_img * 255.0).clip(0, 255).astype('uint8')
            cv2.imwrite(os.path.join(output_dir, f), final_res)
            torch.cuda.empty_cache()

In [ ]:
run_ntire_inference(MODEL_PATH, PAIR_INPUT_DIR, 'pair_results')

Processing T1_Pair_input:   0%|          | 0/26 [00:00<?, ?it/s]

In [ ]:
run_ntire_inference(MODEL_PATH, UNPAIR_INPUT_DIR, 'unpair_results')

Processing Unpair:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
shutil.make_archive('submission_pair', 'zip', 'pair_results')
shutil.make_archive('submission_unpair', 'zip', 'unpair_results')

'/content/submission_unpair.zip'

In [ ]:
generate_submission_zip('best_model.pth', r'/content/drive/MyDrive/val_input')

Processing:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
import shutil
from google.colab import files

# 1. Zip the folder (this creates 'results.zip')
shutil.make_archive('results', 'zip', 'temp_results')

# 2. Download it to your laptop
files.download('results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>